# Heart Disease Continuous Risk Score
**Target:** `risk_score = num / 4` — a continuous value in `[0, 1]` derived from the UCI `num` field.  
**Primary dataset:** `data/processed.cleveland.data` (303 rows, 14 columns).  
**External validation:** Hungarian, Switzerland, VA Long Beach.

## Section 1 — Imports, Constants, and Dataset Loading

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from heart_risk.config import *
from heart_risk.data import load_cleveland, load_all, summary
from heart_risk.eda import run_full_eda
from heart_risk.preprocess import build_preprocessor, split_cleveland, get_feature_names
from heart_risk.models import train_baselines, train_gradient_boosting, select_best, save_pipeline
from heart_risk.evaluate import (
    evaluate_split, evaluate_external, compute_metrics,
    risk_band_confusion_matrix, binary_auc, save_metrics, clip_predictions
)
from heart_risk.explain import run_explainability
from heart_risk.visualize import plot_pred_vs_actual, plot_residuals, plot_risk_by_demographics, progression_table

print('Imports OK')

In [ ]:
# Load all datasets
df_cleveland = load_cleveland()
all_dfs = load_all()
external_dfs = {k: v for k, v in all_dfs.items() if k != 'cleveland'}
print('Cleveland shape:', df_cleveland.shape)
print('First rows:')
df_cleveland.head()

## Section 2 — Data Understanding

In [ ]:
summary(df_cleveland, 'Cleveland')
print('\nDtypes:')
print(df_cleveland.dtypes)
print('\nDescriptive stats:')
df_cleveland.describe()

In [ ]:
print('Missing value table:')
missing = df_cleveland.isnull().sum()
missing_pct = (missing / len(df_cleveland) * 100).round(2)
pd.DataFrame({'count': missing, 'pct': missing_pct})[missing > 0]

In [ ]:
print('External hospital summaries:')
for name, df in external_dfs.items():
    summary(df, name)
    if name == 'hungarian':
        print('  NOTE: Hungarian processed file contains only num=0 and num=1;')
        print('        its continuous target range is narrower than Cleveland.')

## Section 3 — Exploratory Data Analysis

In [ ]:
eda_paths = run_full_eda(df_cleveland)
for p in eda_paths:
    print(p)

In [ ]:
# Display plots inline
from IPython.display import Image, display
for p in eda_paths:
    display(Image(str(p)))

## Section 4 — Preprocessing

**Design choices:**
- `?` values in the raw CSV are read as `NaN` (handled by `pd.read_csv(na_values='?')`).
- `ca` (number of major vessels): **median imputation** — the distribution is right-skewed with a mode of 0, but median imputation is more robust than mode when zero is already the most common value and the missing rate is small (4/303 ≈ 1.3 %).
- `thal` (thalassemia type): **mode imputation** — this is a true categorical variable; replacing with the most frequent code (3 = normal) is semantically appropriate and introduces minimal distributional bias.
- **Regression framing via `num / 4`**: The `num` field encodes severity on an ordinal 0–4 scale. Dividing by 4 produces a proportional risk score in `[0, 1]` that preserves rank order and allows continuous regression instead of five-class ordinal classification, which would require more samples to train reliably.

In [ ]:
X_train, X_test, y_train, y_test = split_cleveland(df_cleveland)
print('Train size:', X_train.shape, '| Test size:', X_test.shape)

# Verify risk_score range
assert df_cleveland[TARGET].between(0, 1).all(), 'risk_score out of [0,1]'
print('risk_score range: [{:.2f}, {:.2f}]'.format(
    df_cleveland[TARGET].min(), df_cleveland[TARGET].max()))

# Preview preprocessor
prep = build_preprocessor()
prep.fit(X_train)
X_train_t = prep.transform(X_train)
X_test_t = prep.transform(X_test)
feature_names = get_feature_names(prep)
print('Transformed feature count:', len(feature_names))
print('Any NaN in train?', np.isnan(X_train_t).any())
print('Any NaN in test?', np.isnan(X_test_t).any())

## Section 5 — Model Training

In [ ]:
print('=== Baseline Models ===')
baseline_results, baseline_fitted = train_baselines(X_train, y_train)
pd.DataFrame(baseline_results)[['model','cv_mae','cv_mae_std','cv_r2']]

In [ ]:
print('=== Gradient Boosting Models (RandomizedSearchCV) ===')
gb_results, gb_fitted = train_gradient_boosting(X_train, y_train)
pd.DataFrame(gb_results)[['model','cv_mae']]

In [ ]:
best_name, best_pipeline, all_results = select_best(
    baseline_results, gb_results, baseline_fitted, gb_fitted
)
save_pipeline(best_pipeline)
pd.DataFrame(all_results).sort_values('cv_mae')

## Section 6 — Evaluation

In [ ]:
# Reconstruct test-set rows from original df (need num column for AUC)
df_test_raw = df_cleveland.loc[X_test.index]
test_metrics = evaluate_split(best_pipeline, X_test, y_test, df_test_raw, tag='cleveland_test')
save_metrics(test_metrics, 'cleveland_test_metrics.json')

In [ ]:
display(Image(str(FIGURES_DIR / 'cm_cleveland_test.png')))

In [ ]:
print('\n=== External Hospital Validation ===')
external_metrics = evaluate_external(best_pipeline, external_dfs)

## Section 7 — Explainability

In [ ]:
run_explainability(best_pipeline, X_train, X_test)

In [ ]:
for fname in ['07_feature_importance.png','08_shap_summary.png',
              '09_shap_waterfall.png','11_pdp.png']:
    p = FIGURES_DIR / fname
    if p.exists():
        display(Image(str(p)))

## Section 8 — Risk Profile Visualization

In [ ]:
y_pred_test = clip_predictions(best_pipeline.predict(X_test))
plot_pred_vs_actual(y_test, y_pred_test, tag='cleveland_test')
plot_residuals(y_test, y_pred_test, tag='cleveland_test')

for fname in ['12_pred_vs_actual_cleveland_test.png', '13_residuals_cleveland_test.png']:
    display(Image(str(FIGURES_DIR / fname)))

In [ ]:
# Full Cleveland dataset predictions for demographic analysis
y_pred_all = clip_predictions(best_pipeline.predict(df_cleveland[ALL_FEATURES]))
plot_risk_by_demographics(df_cleveland, y_pred_all)
display(Image(str(FIGURES_DIR / '14_risk_by_demographics.png')))

In [ ]:
df_prog = progression_table(best_pipeline, df_cleveland.loc[X_train.index])
df_prog

## Final Interpretation

The model converts the ordinal `num` severity scale into a continuous risk score that
generalises reasonably across the four UCI hospital cohorts. Key observations:

1. **Feature importance**: `ca`, `thal`, `cp`, and `oldpeak` consistently rank as the top predictors — consistent with clinical literature on coronary artery disease.
2. **External validation caveat**: The Hungarian processed file contains only `num ∈ {0, 1}`, so its effective `risk_score` range is `{0.00, 0.25}`. Metrics on this dataset reflect a narrower target distribution and should be interpreted accordingly.
3. **Imputation justification**: Median imputation for `ca` and mode imputation for `thal` introduce minimal bias given the low missingness rate (< 2 % in Cleveland). The preprocessing is fitted exclusively on the Cleveland training fold to avoid leakage.
4. **Regression framing**: Treating `num / 4` as a continuous target allows gradient boosting to exploit ordinal structure without the label-sparsity problem inherent to five-class ordinal classifiers on a 303-sample dataset.